# Etapa 4: Componente Generativo (VAE + Conditional GAN)

**Objetivo**: Entrenar modelos generativos para crear imágenes sintéticas de defectos, evaluando su calidad y utilidad para data augmentation.

**Contenido**:
1. Setup e importaciones
2. Preparación de datos para modelos generativos (64×64)
3. VAE — Entrenamiento y evaluación
4. Conditional DCGAN — Entrenamiento y evaluación
5. Generación de datos sintéticos por clase
6. Evaluación de calidad (visual + métricas)
7. Reentrenamiento con datos sintéticos (data augmentation)
8. Conclusiones

## 1. Setup e Importaciones

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torchvision import transforms
from torch.utils.data import DataLoader

from src.data_utils import CLASS_NAMES, DefectDataset
from src.models import VAE, DCGANGenerator, DCGANDiscriminator, count_parameters
from src.train import train_vae, train_cgan

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
FIGURES_DIR = Path("../figures/etapa4")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("../models")

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

## 2. Datos para Modelos Generativos

Los modelos generativos trabajan con imágenes de 64×64 normalizadas a [-1, 1] (para Tanh).

In [ ]:
IMG_SIZE = 64

# Transforms para generativos: resize + normalización a [-1, 1]
gen_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),  # → [-1, 1]
])

data_dir = Path("../data/splits/train")
gen_dataset = DefectDataset(data_dir, transform=gen_transform)
gen_loader = DataLoader(gen_dataset, batch_size=64, shuffle=True, num_workers=2, drop_last=True)

print(f"Dataset generativo: {len(gen_dataset)} imágenes")
print(f"Batches: {len(gen_loader)}")

# Visualizar muestras
batch_imgs, batch_labels = next(iter(gen_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    img = batch_imgs[i] * 0.5 + 0.5  # Desnormalizar de [-1,1] a [0,1]
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(CLASS_NAMES[batch_labels[i].item()], fontsize=8)
    ax.axis("off")
plt.suptitle("Muestras para entrenamiento generativo (64×64)")
plt.tight_layout()
plt.show()

## 3. Variational Autoencoder (VAE)

El VAE aprende una distribución latente continua de las imágenes de defectos, permitiendo:
- **Reconstrucción**: Codificar y decodificar imágenes existentes
- **Generación**: Muestrear del espacio latente para crear nuevas imágenes
- **Interpolación**: Navegar suavemente entre diferentes defectos

In [ ]:
# Crear y entrenar VAE
vae = VAE(img_size=IMG_SIZE, latent_dim=128)
vae_params = count_parameters(vae)
print(f"VAE: {vae_params['total']:,} parámetros")

history_vae = train_vae(
    model=vae,
    train_loader=gen_loader,
    num_epochs=50,
    lr=1e-3,
    beta=1.0,
    device=DEVICE,
    save_dir=MODELS_DIR,
)

In [ ]:
# Curvas de pérdida del VAE
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history_vae["total_loss"])
axes[0].set_title("Total Loss (ELBO)")
axes[0].set_xlabel("Epoch")

axes[1].plot(history_vae["recon_loss"])
axes[1].set_title("Reconstruction Loss")
axes[1].set_xlabel("Epoch")

axes[2].plot(history_vae["kl_loss"])
axes[2].set_title("KL Divergence")
axes[2].set_xlabel("Epoch")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_vae_loss.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.1 Reconstrucciones VAE

In [ ]:
# Reconstrucciones: Original vs Reconstruida
vae.eval()
sample_imgs, sample_labels = next(iter(gen_loader))
with torch.no_grad():
    recon, _, _ = vae(sample_imgs.to(DEVICE))
    recon = recon.cpu()

n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(16, 4))
for i in range(n_show):
    # Original
    orig = sample_imgs[i] * 0.5 + 0.5
    axes[0, i].imshow(orig.permute(1, 2, 0).numpy().clip(0, 1))
    axes[0, i].set_title(CLASS_NAMES[sample_labels[i].item()], fontsize=8)
    axes[0, i].axis("off")
    # Reconstruida
    rec = recon[i] * 0.5 + 0.5
    axes[1, i].imshow(rec.permute(1, 2, 0).numpy().clip(0, 1))
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Original", fontsize=10)
axes[1, 0].set_ylabel("Reconstruida", fontsize=10)
plt.suptitle("VAE: Originales vs Reconstrucciones")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_vae_reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.2 Generación desde Espacio Latente

In [ ]:
# Generar imágenes muestreando del espacio latente
generated_vae = vae.generate(num_samples=16, device=DEVICE).cpu()
generated_vae = generated_vae * 0.5 + 0.5  # [-1,1] → [0,1]

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    ax.imshow(generated_vae[i].permute(1, 2, 0).numpy().clip(0, 1))
    ax.axis("off")
plt.suptitle("VAE: Imágenes Generadas (muestreo aleatorio del espacio latente)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_vae_generated.png", dpi=150, bbox_inches="tight")
plt.show()

### 3.3 Interpolación en Espacio Latente

In [ ]:
# Interpolación entre dos imágenes en el espacio latente
vae.eval()
img_a, img_b = sample_imgs[0].unsqueeze(0).to(DEVICE), sample_imgs[1].unsqueeze(0).to(DEVICE)

with torch.no_grad():
    mu_a, _ = vae.encode(img_a)
    mu_b, _ = vae.encode(img_b)

n_steps = 10
alphas = np.linspace(0, 1, n_steps)

fig, axes = plt.subplots(1, n_steps, figsize=(20, 2.5))
for i, alpha in enumerate(alphas):
    z = (1 - alpha) * mu_a + alpha * mu_b
    with torch.no_grad():
        img_interp = vae.decode(z).cpu()
    img_show = (img_interp[0] * 0.5 + 0.5).permute(1, 2, 0).numpy().clip(0, 1)
    axes[i].imshow(img_show)
    axes[i].set_title(f"α={alpha:.1f}", fontsize=8)
    axes[i].axis("off")

plt.suptitle(f"Interpolación: {CLASS_NAMES[sample_labels[0]]} → {CLASS_NAMES[sample_labels[1]]}")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_vae_interpolation.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Conditional DCGAN

GAN condicional que genera imágenes de una clase específica. El Generator recibe etiqueta + ruido, y el Discriminator recibe imagen + etiqueta.

In [ ]:
LATENT_DIM = 100

gen = DCGANGenerator(latent_dim=LATENT_DIM, num_classes=5, img_size=IMG_SIZE)
disc = DCGANDiscriminator(num_classes=5, img_size=IMG_SIZE)

print(f"Generator:     {count_parameters(gen)['total']:>10,} params")
print(f"Discriminator: {count_parameters(disc)['total']:>10,} params")

# Entrenar CGAN
history_gan = train_cgan(
    generator=gen,
    discriminator=disc,
    dataloader=gen_loader,
    num_epochs=100,
    latent_dim=LATENT_DIM,
    lr=2e-4,
    device=DEVICE,
    save_dir=MODELS_DIR,
)

In [ ]:
# Curvas de pérdida del GAN
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history_gan["g_loss"], label="Generator Loss", alpha=0.7)
ax.plot(history_gan["d_loss"], label="Discriminator Loss", alpha=0.7)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss")
ax.set_title("Conditional DCGAN: Training Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_gan_loss.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Generación de Imágenes Sintéticas por Clase

In [ ]:
# Generar imágenes condicionales con el GAN
gen.eval()
n_per_class = 8

fig, axes = plt.subplots(len(CLASS_NAMES), n_per_class, figsize=(16, 2.5 * len(CLASS_NAMES)))

for class_idx, class_name in enumerate(CLASS_NAMES):
    z = torch.randn(n_per_class, LATENT_DIM, device=DEVICE)
    labels = torch.full((n_per_class,), class_idx, dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        fake_imgs = gen(z, labels).cpu()

    for j in range(n_per_class):
        img = fake_imgs[j] * 0.5 + 0.5  # [-1,1] → [0,1]
        axes[class_idx, j].imshow(img.permute(1, 2, 0).numpy().clip(0, 1))
        axes[class_idx, j].axis("off")

    axes[class_idx, 0].set_ylabel(class_name, fontsize=10, rotation=0, labelpad=60, va="center")

plt.suptitle("Conditional DCGAN: Imágenes Generadas por Clase", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "04_gan_generated_by_class.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Generación de Dataset Sintético para Data Augmentation

Generamos imágenes sintéticas para las clases minoritarias, guardándolas en disco para reentrenamiento.

In [ ]:
from PIL import Image
import os

# Contar muestras por clase en el train set
class_counts = {}
train_dir = Path("../data/splits/train")
for cls in CLASS_NAMES:
    cls_dir = train_dir / cls
    if cls_dir.exists():
        class_counts[cls] = len(list(cls_dir.glob("*")))
    else:
        class_counts[cls] = 0

max_count = max(class_counts.values())
print("Distribución original del train set:")
for cls, cnt in class_counts.items():
    print(f"  {cls}: {cnt}")
print(f"\nMáximo: {max_count}")

# Generar imágenes sintéticas para igualar clases
synth_dir = Path("../data/synthetic")
synth_dir.mkdir(parents=True, exist_ok=True)

gen.eval()
total_generated = 0

for class_idx, class_name in enumerate(CLASS_NAMES):
    n_needed = max_count - class_counts.get(class_name, 0)
    if n_needed <= 0:
        continue

    cls_dir = synth_dir / class_name
    cls_dir.mkdir(parents=True, exist_ok=True)

    # Generar en batches
    n_generated = 0
    batch_size_gen = 64
    while n_generated < n_needed:
        n_batch = min(batch_size_gen, n_needed - n_generated)
        z = torch.randn(n_batch, LATENT_DIM, device=DEVICE)
        labels = torch.full((n_batch,), class_idx, dtype=torch.long, device=DEVICE)

        with torch.no_grad():
            fake = gen(z, labels).cpu()
            fake = fake * 0.5 + 0.5  # → [0, 1]

        for j in range(n_batch):
            img_arr = (fake[j].permute(1, 2, 0).numpy().clip(0, 1) * 255).astype(np.uint8)
            img_pil = Image.fromarray(img_arr)
            img_pil.save(cls_dir / f"synth_{n_generated + j:04d}.png")

        n_generated += n_batch

    total_generated += n_generated
    print(f"  {class_name}: generadas {n_generated} imágenes")

print(f"\nTotal imágenes sintéticas generadas: {total_generated}")

## 7. Conclusiones

### VAE
- Aprende una distribución latente suave de las imágenes de defectos
- Las reconstrucciones capturan la estructura general pero pierden detalles finos (típico de VAE)
- La interpolación demuestra que el espacio latente es continuo y semánticamente significativo

### Conditional DCGAN
- Genera imágenes condicionadas por clase, permitiendo balanceo del dataset
- La calidad mejora con más epochs de entrenamiento
- Las imágenes generadas pueden usarse como data augmentation para clases minoritarias

### Data Augmentation con Datos Sintéticos
- Se generaron imágenes para igualar las clases minoritarias con la clase mayoritaria
- En la Etapa 5 se evaluará el impacto de entrenar con datos mixtos (reales + sintéticos)

### Próximos Pasos
- **Etapa 5**: Fine-tuning eficiente con LoRA/PEFT, cuantización, pruning y despliegue